In [ ]:
import os,sys,subprocess,shutil,json,time,urllib.request,urllib.parse,zipfile
from pathlib import Path

REPO="https://github.com/Phantom220-hub/Nyxvault.git"
BRANCH="main"
IS_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
ROOT = Path("/kaggle/working") if IS_KAGGLE else Path("/content")
ROOT.mkdir(parents=True, exist_ok=True)
WORK=ROOT/"nyxvault"
OUT=ROOT/"nyxvault-output"
OUT.mkdir(parents=True, exist_ok=True)

# Kaggle has no direct Google Drive mount. Restore the full checkpoint bundle
# from a Kaggle Input dataset containing nyxvault-training-output.zip or
# the extracted nyxvault-output folder.
if IS_KAGGLE:
    input_root = Path("/kaggle/input")
    restored = False
    archive_candidates = sorted(input_root.rglob("nyxvault-training-output.zip")) if input_root.exists() else []
    for archive_path in archive_candidates:
        try:
            with zipfile.ZipFile(archive_path) as z:
                root_resolved = ROOT.resolve()
                for member in z.infolist():
                    destination = (ROOT / member.filename).resolve()
                    if destination != root_resolved and root_resolved not in destination.parents:
                        raise RuntimeError("Архив содержит недопустимый путь.")
                z.extractall(ROOT)
            restored = (OUT/"checkpoints").is_dir()
            if restored:
                print("Checkpoint-архив восстановлен из Kaggle Input:", archive_path)
                break
        except Exception as error:
            print("Не удалось восстановить архив", archive_path, repr(error))

    if not restored and input_root.exists():
        input_checkpoints = list(input_root.rglob("step_*.pt"))
        if input_checkpoints:
            def _step_number(path):
                try:
                    return int(path.stem.split("_")[1])
                except (IndexError, ValueError):
                    return -1
            latest_input = max(input_checkpoints, key=_step_number)
            input_output = latest_input.parent.parent
            shutil.copytree(input_output, OUT, dirs_exist_ok=True)
            restored = True
            print("Checkpoint-папка восстановлена из Kaggle Input:", input_output)

    if not restored or not list((OUT/"checkpoints").glob("step_*.pt")):
        raise RuntimeError(
            "В Kaggle Input не найден архив nyxvault-training-output.zip "
            "или папка nyxvault-output с checkpoint. Сначала добавьте полный "
            "nyxvault-output из Google Drive как Kaggle Dataset. "
            "Один step_600.pt без tokenizer.json и processed_data недостаточен."
        )

# Persist the full base-model state between temporary Colab sessions.

# Persist the full base-model state between temporary Colab sessions.
DRIVE_ROOT = None
DRIVE_OUT = None
if not IS_KAGGLE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/NYXVAULT")
    DRIVE_OUT = DRIVE_ROOT/"nyxvault-output"
    local_steps = list((OUT/"checkpoints").glob("step_*.pt")) if (OUT/"checkpoints").is_dir() else []
    if DRIVE_OUT.is_dir() and not local_steps:
        print("Восстанавливаем предыдущие результаты из Google Drive:", DRIVE_OUT)
        shutil.copytree(DRIVE_OUT, OUT, dirs_exist_ok=True)

CHECKPOINT_DIR=OUT/"checkpoints"
checkpoint_candidates=[]
for candidate in CHECKPOINT_DIR.glob("step_*.pt") if CHECKPOINT_DIR.is_dir() else []:
    try:
        checkpoint_candidates.append((int(candidate.stem.split("_")[1]), candidate))
    except (IndexError, ValueError):
        continue
BASE_STEP, BASE_CHECKPOINT = max(checkpoint_candidates, key=lambda item:item[0]) if checkpoint_candidates else (0, None)
REUSE_BASE_DATA = (
    BASE_CHECKPOINT is not None
    and (OUT/"tokenizer.json").is_file()
    and (OUT/"processed_data/train.txt").is_file()
    and (OUT/"processed_data/validation.txt").is_file()
)
print("Платформа:", "Kaggle" if IS_KAGGLE else "Google Colab")
print("Рабочая папка:", ROOT)
print("GPU будет проверена после загрузки исходников.")
if BASE_CHECKPOINT:
    print("Найден checkpoint:", BASE_CHECKPOINT.name, "— шаг", BASE_STEP)
if REUSE_BASE_DATA:
    print("Подготовленные данные и токенизатор будут повторно использованы; Wikipedia скачиваться заново не будет.")
if IS_KAGGLE and not REUSE_BASE_DATA:
    raise RuntimeError(
        "В Kaggle восстановлен checkpoint, но отсутствует совместимый tokenizer.json "
        "или processed_data. Загрузите полный nyxvault-output, а не только веса."
    )

# Fetch one clean copy of current main. The artifact folder OUT is kept separate.
if WORK.exists():
    shutil.rmtree(WORK)
clone = subprocess.run(
    ["git","clone","--depth","1","--branch",BRANCH,REPO,str(WORK)],
    capture_output=True,text=True,timeout=120
)
if clone.returncode != 0:
    print("Git clone не сработал:", clone.stderr.strip()[-1200:] or clone.stdout.strip()[-1200:])
    if WORK.exists(): shutil.rmtree(WORK)
    archive_url=f"https://codeload.github.com/Phantom220-hub/Nyxvault/zip/refs/heads/{BRANCH}"
    archive_path=ROOT/"nyxvault-source.zip"
    try:
        req=urllib.request.Request(archive_url,headers={"User-Agent":"NYXVAULT-training"})
        with urllib.request.urlopen(req,timeout=90) as response, open(archive_path,"wb") as target:
            shutil.copyfileobj(response,target)
        with zipfile.ZipFile(archive_path) as z:
            prefix="Nyxvault-main/"
            for member in z.infolist():
                name=member.filename
                if not name.startswith(prefix): continue
                relative=Path(name[len(prefix):])
                if not relative.parts: continue
                destination=(WORK/relative).resolve()
                if WORK.resolve() not in destination.parents and destination != WORK.resolve():
                    raise RuntimeError("Недопустимый путь в архиве GitHub")
                if member.is_dir(): destination.mkdir(parents=True,exist_ok=True)
                else:
                    destination.parent.mkdir(parents=True,exist_ok=True)
                    with z.open(member) as src_file, open(destination,"wb") as dst_file:
                        shutil.copyfileobj(src_file,dst_file)
        if not (WORK/"training/train.py").is_file():
            raise RuntimeError("Архив скачан, но структура репозитория неполная")
        print("Исходники загружены через GitHub ZIP.")
    except Exception as download_error:
        if WORK.exists(): shutil.rmtree(WORK)
        raise RuntimeError("Не удалось загрузить Nyxvault ни через git, ни через GitHub ZIP. "+repr(download_error)) from download_error
finally_archive=ROOT/"nyxvault-source.zip"
if finally_archive.exists(): finally_archive.unlink()
os.chdir(WORK)

# Install project dependencies and require the GPU for a base-training run.
import torch
subprocess.check_call([sys.executable,"-m","pip","install","-q","tokenizers","numpy","onnx","onnxruntime","onnxscript","requests"])
print("GPU:",torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT FOUND")
if not torch.cuda.is_available():
    raise RuntimeError("Включите GPU: Settings → Accelerator → GPU, затем выполните ячейку снова.")

if REUSE_BASE_DATA:
    # Reuse the exact same processed corpus and tokenizer as the saved checkpoint.
    shutil.copy2(OUT/"tokenizer.json", WORK/"artifacts/tokenizer.json")
    shutil.copytree(OUT/"processed_data", WORK/"training/data/processed", dirs_exist_ok=True)
    if (OUT/"attributions.jsonl").is_file():
        shutil.copy2(OUT/"attributions.jsonl", WORK/"training/data/attributions.jsonl")
    print("Повторно использованы сохранённые данные и токенизатор.")
else:
    raw=WORK/"training/data/raw"
    raw.mkdir(parents=True,exist_ok=True)
    seed=WORK/"training/data/programming_seed.jsonl"
    project_seed=WORK/"training/data/code_projects_seed.jsonl"
    if seed.is_file(): shutil.copy2(seed,raw/"programming_seed.jsonl")
    if project_seed.is_file(): shutil.copy2(project_seed,raw/"code_projects_seed.jsonl")
    print("Programming seed examples:",sum(1 for _ in open(raw/"programming_seed.jsonl",encoding="utf-8")) if (raw/"programming_seed.jsonl").exists() else 0)
    print("Code and game-project examples:",sum(1 for _ in open(raw/"code_projects_seed.jsonl",encoding="utf-8")) if (raw/"code_projects_seed.jsonl").exists() else 0)

    topics={
     "ru":[
      "программирование","язык программирования","Python","язык программирования Python","C++","язык программирования C","JavaScript","TypeScript","Java","Kotlin","C Sharp","Rust","PHP","GDScript","Godot","HTML","CSS","веб-разработка","разработка программного обеспечения","алгоритм","структура данных","массив","связный список","стек данных","очередь данных","хеш-таблица","дерево поиска","граф теория","сортировка алгоритм","поиск алгоритм","рекурсия","объектно-ориентированное программирование","функциональное программирование","компилятор","интерпретатор","отладка программ","тестирование программного обеспечения","система контроля версий","Git","GitHub","SQL","система управления базами данных","API","HTTP","JSON","операционная система","Linux","разработка Android","машинное обучение","нейронная сеть","кибербезопасность",
      "информационные технологии","математика","наука","образование","компьютерная сеть","робототехника"
     ],
     "tg":[
      "барномасозӣ","забони барномасозӣ","Python","JavaScript","C++","HTML","CSS","алгоритм","сохтори додаҳо","пойгоҳи додаҳо","шабакаи компютерӣ","амнияти киберӣ","зеҳни сунъӣ","омӯзиши мошинӣ","робототехника",
      "илм","маориф","технологияи иттилоотӣ","забони тоҷикӣ","математика"
     ],
     "en":[
      "computer programming","programming language","Python programming language","JavaScript","TypeScript","C++","Java programming language","Kotlin programming language","Rust programming language","HTML","CSS","GDScript","Godot Engine","software engineering","software testing","debugging","algorithm","data structure","array data structure","linked list","stack data structure","queue data structure","hash table","binary tree","graph theory","sorting algorithm","search algorithm","recursion","object-oriented programming","functional programming","compiler","interpreter","version control","Git","GitHub","database","SQL","application programming interface","HTTP","JSON","operating system","Linux","Android software development","machine learning","artificial neural network","cybersecurity","computer network","web development","application software","computer science",
      "mathematics","science","education","technology"
     ]
    }
    def api(lang,params):
        url="https://"+lang+".wikipedia.org/w/api.php?"+urllib.parse.urlencode(params)
        req=urllib.request.Request(url,headers={"User-Agent":"NYXVAULT educational research bot"})
        with urllib.request.urlopen(req,timeout=35) as response:
            return json.loads(response.read().decode())
    attrs=[]
    seen=set()
    for lang,terms in topics.items():
        for term in terms:
            try:
                hits=api(lang,{"action":"query","list":"search","srsearch":term,"srlimit":6,"format":"json"}).get("query",{}).get("search",[])
                for hit in hits:
                    title=hit["title"]
                    if (lang,title) in seen: continue
                    seen.add((lang,title))
                    pages=api(lang,{"action":"query","prop":"extracts|revisions","explaintext":1,"rvprop":"ids","titles":title,"format":"json"}).get("query",{}).get("pages",{})
                    page=next(iter(pages.values()))
                    body=page.get("extract","").strip()
                    if len(body)<600: continue
                    name="".join(ch if ch.isalnum() else "_" for ch in title)[:80]
                    (raw/f"{lang}_{name}.txt").write_text(body,encoding="utf-8")
                    attrs.append({"lang":lang,"title":title,"url":"https://"+lang+".wikipedia.org/wiki/"+urllib.parse.quote(title.replace(" ","_")),"revision_id":(page.get("revisions") or [{}])[0].get("revid"),"retrieved_utc":time.strftime("%Y-%m-%dT%H:%M:%SZ",time.gmtime()),"license":"CC BY-SA 4.0 / GFDL; verify page exceptions","modified":"plain-text extraction; cleaned"})
                    time.sleep(1.0)
            except Exception as error:
                print("Skipped",lang,term,str(error)[:100])
    (WORK/"training/data/attributions.jsonl").write_text(
        "".join(json.dumps(item,ensure_ascii=False)+"\n" for item in attrs),
        encoding="utf-8"
    )
    print("Wikimedia pages:",len(attrs))
    if len(attrs)<5:
        raise RuntimeError("Слишком мало текстов Wikipedia (<5). Проверьте интернет; seed-файлы проекта доступны, но для этого этапа требуется минимум 5 страниц.")
    print("Wikipedia страниц получено:",len(attrs),"(429 означает ограничение запросов, а не сбой обучения).")
    subprocess.run([sys.executable,"training/prepare_data.py","--input","training/data/raw","--knowledge","data/knowledge.jsonl","--out","training/data/processed"],check=True)
    subprocess.run([sys.executable,"training/train_tokenizer.py","--input","training/data/processed/train.txt","--output","artifacts/tokenizer.json","--vocab-size","8000"],check=True)
    subprocess.run([sys.executable,"training/pipeline_smoke_test.py","--data","training/data/processed","--tokenizer","artifacts/tokenizer.json","--block-size","32","--steps","3"],check=True)
    shutil.copy2("artifacts/tokenizer.json",OUT/"tokenizer.json")
    shutil.copytree("training/data/processed",OUT/"processed_data",dirs_exist_ok=True)
    shutil.copy("training/data/attributions.jsonl",OUT/"attributions.jsonl")
    shutil.copy("training/data/processed/manifest.json",OUT/"data_manifest.json")

from tokenizers import Tokenizer
from training.model import NYXVAULTGPT
subprocess.run([sys.executable,"training/smoke_test.py"],check=True)
vocab=Tokenizer.from_file("artifacts/tokenizer.json").get_vocab_size()
probe_cfg={"d_model":256,"n_heads":8,"n_layers":6,"block_size":512,"dropout":0.1}
probe=NYXVAULTGPT(vocab,probe_cfg)
print("Параметров обучаемой модели:",f"{probe.parameter_count:,}")

# Automatically resume from the newest full base checkpoint, if one exists.
checkpoint_candidates=[]
for candidate in (OUT/"checkpoints").glob("step_*.pt") if (OUT/"checkpoints").is_dir() else []:
    try:
        checkpoint_candidates.append((int(candidate.stem.split("_")[1]),candidate))
    except (IndexError,ValueError):
        continue
RESUME_FROM=str(max(checkpoint_candidates,key=lambda item:item[0])[1]) if checkpoint_candidates else ""
if RESUME_FROM:
    print("Продолжаем с checkpoint:",Path(RESUME_FROM).name)
else:
    print("Запускаем базовое обучение с нуля.")

MAX_NEW_STEPS = 600 if IS_KAGGLE else 100
print("Новых шагов обучения в этом запуске:", MAX_NEW_STEPS)
cmd=[sys.executable,"training/train.py","--data","training/data/processed","--steps","5000","--max-steps",str(MAX_NEW_STEPS),"--batch-size","8","--grad-accum-steps","4","--warmup-steps","150","--precision","auto","--block-size","512","--save-every","500","--log-interval","10","--save-first-checkpoint","100","--out",str(OUT/"checkpoints")]
if RESUME_FROM:
    cmd += ["--resume",RESUME_FROM]
subprocess.run(cmd,check=True)

shutil.copy2("artifacts/tokenizer.json",OUT/"tokenizer.json")
shutil.copytree("training/data/processed",OUT/"processed_data",dirs_exist_ok=True)
if (WORK/"training/data/attributions.jsonl").is_file():
    shutil.copy2("training/data/attributions.jsonl",OUT/"attributions.jsonl")
shutil.copy2("training/data/processed/manifest.json",OUT/"data_manifest.json")
with open(OUT/"evaluation.txt","w",encoding="utf-8") as report:
    subprocess.run([sys.executable,"training/evaluate.py","--checkpoint",str(OUT/"checkpoints/best.pt"),"--tokenizer",str(OUT/"tokenizer.json"),"--data","training/data/processed/validation.txt"],stdout=report,check=True)
try:
    subprocess.run([sys.executable,"training/export_mobile.py","--checkpoint",str(OUT/"checkpoints/best.pt"),"--out",str(OUT/"nyxvault.onnx"),"--quantize"],check=True)
except Exception as error:
    print("ONNX экспорт пропущен:",repr(error))

# A single portable archive, plus a persistent Drive copy for the next stage.
archive=ROOT/"nyxvault-training-output.zip"
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for file in OUT.rglob("*"):
        if file.is_file():
            z.write(file,file.relative_to(ROOT))
if DRIVE_ROOT is not None:
    DRIVE_ROOT.mkdir(parents=True,exist_ok=True)
    shutil.copytree(OUT,DRIVE_OUT,dirs_exist_ok=True)
    shutil.copy2(archive,DRIVE_ROOT/archive.name)
    print("Checkpoint, tokenizer, processed corpus and archive saved to Google Drive:",DRIVE_ROOT)
print("ГОТОВО: архив:",archive)
print("Следующий шаг: открыть SFT notebook — он продолжит работу с последнего checkpoint из Google Drive.")

## Публикация базовой модели в GitHub Releases (необязательно)

Если в Colab Secrets сохранён `NYXVAULT_GITHUB_TOKEN` с правом создания Releases и загрузки assets в `Phantom220-hub/Nyxvault`, архив с текущими checkpoint, токенизатором, подготовленными данными и ONNX будет прикреплён к Release этого же репозитория. Без секрета архив останется в Google Drive. Никогда не вставляйте токен в код или чат.

In [ ]:
if IS_KAGGLE:
    print("Kaggle: автоматическая публикация GitHub Release пропущена.")
    print("Архив с checkpoint и данными сохранён в:", archive)
else:
    from google.colab import userdata
    token = None
    try:
        token = userdata.get("NYXVAULT_GITHUB_TOKEN")
    except Exception:
        token = None

    latest_steps = []
    for checkpoint in (OUT / "checkpoints").glob("step_*.pt"):
        try:
            latest_steps.append(int(checkpoint.stem.split("_")[1]))
        except (IndexError, ValueError):
            pass
    if token and latest_steps:
        env = os.environ.copy()
        env["GITHUB_TOKEN"] = token
        current_step = max(latest_steps)
        subprocess.run([
            sys.executable, "training/publish_release_assets.py",
            "--repo", "Phantom220-hub/Nyxvault",
            "--tag", f"nyxvault-base-step-{current_step}",
            "--release-name", f"NYXVAULT base Transformer — step {current_step}",
            "--asset", str(ROOT / "nyxvault-training-output.zip"),
        ], cwd=WORK, env=env, check=True)
    else:
        print("GitHub Release upload пропущен. Архив сохранён локально; добавьте Colab Secret NYXVAULT_GITHUB_TOKEN для публикации.")
